#

#Organização de Dados

O objetivo desse notebook é simplesmente organizar as imagens da pasta de Data para que possamos separar os dados em treino e teste de forma estratégica, já que irei utilizar o pytorch em conjunto com a resnet18 (transfer learning) para resolver esse problema.

In [1]:
import random
import os
import shutil
AI_PATH = r"E:\Projects\Data-Science-Portfolio\08_portfolio_real_vs_ai_image_prediction\predict_image\Data\raw\Ai_generated_dataset"
REAL_PATH = r"E:\Projects\Data-Science-Portfolio\08_portfolio_real_vs_ai_image_prediction\predict_image\Data\raw\real_dataset"

In [2]:
def get_all_images(base_path):
    """
    FUNÇÃO PRA LISTAR TODAS AS IMAGENS DEPENDENDO DO DIRETÓRIO (IA OU REAL)
    """
    image_paths = []
    
    for category in os.listdir(base_path):
        category_path = os.path.join(base_path, category)
        
        if os.path.isdir(category_path):
            for img in os.listdir(category_path):
                full_path = os.path.join(category_path, img)
                image_paths.append(full_path)
    
    return image_paths

In [3]:
real_images = get_all_images(REAL_PATH)
ai_images = get_all_images(AI_PATH)

print("Real:", len(real_images))
print("AI:", len(ai_images))

Real: 745
AI: 250


É possível perceber que o dataset tá altamente desbalanceado, temos uma proporção de 3:1 de imagens reais do que geradas por inteligência artificial. Por esse motivo eu irei colocar peso nas classes (Class Weights), com o intuito de que treinar o modelo pensando que errar IA é mais grave do que errar real.
peso_classe = total_amostras / (n_classes * amostras_da_classe)

In [4]:
total = len(real_images)+len(ai_images)
n_classes = 2

In [5]:
peso_real = total / (n_classes * len(real_images))
peso_ai = total / (n_classes * len(ai_images))
print(f"Peso classe IA: {peso_ai} | Peso classe Real: {peso_real}")

Peso classe IA: 1.99 | Peso classe Real: 0.6677852348993288


Como vamos utilizar pytorch, vou precisar converter esses pesos em tensor. 

Classe 0 - AI

Classe 1 - REAL

In [6]:
import torch 

weights = torch.tensor([peso_ai, peso_real])
weights

tensor([1.9900, 0.6678])

##Organizando as imagens nas pastas de teste, treino e validação

In [7]:
BASE_DIR = r"E:\Projects\Data-Science-Portfolio\08_portfolio_real_vs_ai_image_prediction\predict_image"

In [26]:
def copy_images(image_list, split_name, label):
    """
    FUNÇÃO PARA COPIAR AS IMAGENS DENTRO DAS RESPECTIVAS PASTAS
    """
    
    for img_path in image_list:
        category = os.path.basename(os.path.dirname(img_path))
        
        file_name = f"{category}_{os.path.basename(img_path)}"
        
        destination = os.path.join(
        BASE_DIR,
        "Data",
        split_name,
        label,
        file_name
        )
        
        shutil.copy(img_path, destination)


In [27]:
import random

def split_data(image_list, train_ratio=0.7, val_ratio=0.15):
    """
    FUNÇÃO PARA DIVIDIR DADOS DE FORMA ALEATÓRIA A PARTIR DE SUA CLASSE
    """
    random.shuffle(image_list)
    
    total = len(image_list)
    train_end = int(total * train_ratio)
    val_end = int(total * (train_ratio + val_ratio))
    
    train = image_list[:train_end]
    val = image_list[train_end:val_end]
    test = image_list[val_end:]
    
    return train, val, test

In [28]:
real_train, real_val, real_test = split_data(real_images)
ai_train, ai_val, ai_test = split_data(ai_images)

In [29]:
copy_images(real_train, "train", "real")
copy_images(real_val, "val", "real")
copy_images(real_test, "test", "real")

copy_images(ai_train, "train", "ai")
copy_images(ai_val, "val", "ai")
copy_images(ai_test, "test", "ai")

Implementei split estratificado por classe para evitar data leakage e garantir distribuição consistente!

In [8]:
TRAIN_DIR = r"E:\Projects\Data-Science-Portfolio\08_portfolio_real_vs_ai_image_prediction\predict_image\Data\train"
VAL_DIR = r"E:\Projects\Data-Science-Portfolio\08_portfolio_real_vs_ai_image_prediction\predict_image\Data\val"
TEST_DIR = r"E:\Projects\Data-Science-Portfolio\08_portfolio_real_vs_ai_image_prediction\predict_image\Data\test"

In [9]:
train_images = get_all_images(TRAIN_DIR)
val_images = get_all_images(VAL_DIR)
test_images = get_all_images(TEST_DIR)
#printando o número de imagens de cada diretório
print("Treino: ", len(train_images))
print("Validação: ", len(val_images))
print("Teste: ", len(test_images))

Treino:  696
Validação:  149
Teste:  150


In [10]:
from torchvision import datasets

In [11]:
train_dataset = datasets.ImageFolder(root=TRAIN_DIR)
val_dataset = datasets.ImageFolder(root = VAL_DIR)
test_dataset = datasets.ImageFolder(root = TEST_DIR)

O ImageFolder organiza automaticamente imagens em labels a partir da estrutura de diretórios, mas não realiza split de dados,essa etapa eu implementei manualmente para evitar data leakage.

In [12]:
print(train_dataset.classes)
print(train_dataset.class_to_idx)
print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))
print(weights)
#verificando o label que o ImageFolder deu para cada uma das duas classes e se está tudo certo com o número de imagens!

['ai', 'real']
{'ai': 0, 'real': 1}
696
149
150
tensor([1.9900, 0.6678])


#Transforms (Data Augmentation)

Realizando modificações nas imagens de treino com transforms para uma melhor validação do modelo.

In [13]:
from torchvision import transforms
import torch 

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    
    transforms.ToTensor(),
    
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])





🧠 O que está acontecendo aqui:
Resize → padrão do modelo
Augmentation → robustez
ToTensor → converte e escala (0–1)
Normalize → padrão ImageNet

TRANSFORM DE VALIDAÇÃO/TESTE

In [14]:
transform_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    
    transforms.ToTensor(),
    
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

Conectando com o Dataset

In [15]:
train_dataset = datasets.ImageFolder(root=TRAIN_DIR,transform=transform_train)
val_dataset = datasets.ImageFolder(root = VAL_DIR,transform=transform_eval)
test_dataset = datasets.ImageFolder(root = TEST_DIR, transform=transform_eval)

In [16]:
img, label = train_dataset[0]

print(img.shape)
print(label)

torch.Size([3, 224, 224])
0


#Implementando o DataLoader

In [17]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    dataset= train_dataset,
    batch_size= 32,
    shuffle = True
)

val_loader = DataLoader(
    dataset= val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    dataset= test_dataset,
    batch_size= 32,
    shuffle= False
)

Testando se o loader funcionou

In [18]:
for images, labels in train_loader:
    print(images.shape)
    print(labels.shape)
    break

torch.Size([32, 3, 224, 224])
torch.Size([32])


#Carregando a RESNET18

In [20]:
import torchvision.models as models
import torch.nn as nn 
model = models.resnet18(pretrained = True)
print(model)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

Eu congelo as camadas iniciais porque elas já aprenderam features gerais (bordas, texturas, formas) no ImageNet. Assim, evito overfitting e reduzo custo computacional, focando apenas na adaptação da camada final ao meu problema específico.

In [ ]:
for param in model.parameters(): #congelando todas as camadas
    param.requires_grad = False
    
model.fc = torch.nn.Linear(model.fc.in_features, 2) #cria nova camada que é treinável por padrão

In [25]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

fc.weight
fc.bias


O problema tem duas saídas, vai ser natural para essa classificação multiclasse o uso de uma loss function como a CrossEntropyLoss. Como otimizador, irei utilizar o Adam. Utilizei uma função lambda para garatir que estou filtrando apenas a camada final para o treino.

In [ ]:
loss_fn = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

#Treinando o modelo

Irei também definir o device por aqui (CPU ou GPU). Esse é um procedimento padrão, nesse caso para treinar com a GPU eu precisaria utilizar um ambiente como o google colab!

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


 Devido ao tamanho reduzido do dataset e ao uso de transfer learning com camadas congeladas, o treinamento será realizado com a CPU sem impacto significativo no tempo de execução. Em cenários maiores, o uso de GPU seria essencial para eficiência. Estou rodando o pytorch localmente e não tenho uma gpu da NVIDIA, a minha é da AMD (RX 7800XT).

seguindo a ordem de treino do pytorch:
1. zero_grad()
2. forward
3. loss
4. backward()
5. optimizer.step()

In [31]:
epochs = 1

for epoch in range(epochs):
    model.train()
    for images,labels in train_loader:
        
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        
        loss = loss_fn(outputs, labels)
        
        loss.backward()
        
        optimizer.step()
        
        print(loss.item())
          
        

    
        

0.8179247379302979
0.6161800026893616
0.6968220472335815
0.6392250657081604
0.6417262554168701
0.6565909385681152
0.7368226647377014
0.6223741173744202
0.5921520590782166
0.6511576175689697
0.7331489324569702
0.5478817224502563
0.5930134654045105
0.46998584270477295
0.6349931359291077
0.5976213812828064
0.5774701237678528
0.5176622867584229
0.5455603003501892
0.654845654964447
0.45065730810165405
0.36860769987106323
